TITRE

INTRODUCTION

SOMMAIRE

Installation

In [1]:
!pip install -r requirements.txt
#Modules:
import geopandas as gpd
import pandas as pd
#Fonctions:
import src.clear_data as cd
import src.get_data as gd
import src.analyse_data as ad

Préparation des données

Adresses

In [2]:
url_metro = "https://data.rennesmetropole.fr/explore/dataset/topologie-des-points-darret-de-metro-du-reseau-star/download/?format=geojson"

url_dvf2021 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2021/full.csv.gz"

url_dvf2022 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2022/full.csv.gz"

url_dvf2023 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2023/full.csv.gz"

url_dvf2024 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2024/full.csv.gz"

url_dvf2025 = "https://files.data.gouv.fr/geo-dvf/latest/csv/2025/full.csv.gz"

Import et modifications

In [3]:
#Import
df_dvf_raw2021 = gd.fetch_dvf_api(url_dvf2021)
df_dvf_raw2022 = gd.fetch_dvf_api(url_dvf2022)
df_dvf_raw2023 = gd.fetch_dvf_api(url_dvf2023)
df_dvf_raw2024 = gd.fetch_dvf_api(url_dvf2024)
df_dvf_raw2025 = gd.fetch_dvf_api(url_dvf2025)


--- Récupération DVF (Source miroir stable) ---
--- Récupération DVF (Source miroir stable) ---


/home/onyxia/work/Projet-python-pour-la-data-science/src/get_data.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_rennes = pd.concat(df_rennes_list)


--- Récupération DVF (Source miroir stable) ---


/home/onyxia/work/Projet-python-pour-la-data-science/src/get_data.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_rennes = pd.concat(df_rennes_list)


--- Récupération DVF (Source miroir stable) ---


/home/onyxia/work/Projet-python-pour-la-data-science/src/get_data.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_rennes = pd.concat(df_rennes_list)


--- Récupération DVF (Source miroir stable) ---


/home/onyxia/work/Projet-python-pour-la-data-science/src/get_data.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_rennes = pd.concat(df_rennes_list)


--- Récupération des données Métro via API (Rennes Métropole) ---


Analyse des bases de données en vu de les nettoyer

Analyse des 5 dvfs en vue de la fusion


In [4]:
list_dvf = [
    df_dvf_raw2021, 
    df_dvf_raw2022, 
    df_dvf_raw2023, 
    df_dvf_raw2024, 
    df_dvf_raw2025
]
labels = ["2021", "2022", "2023", "2024", "2025"]
ad.verify_dvf_columns(list_dvf, labels)


--- Diagnostic de cohérence des colonnes ---
Année 2021 : Colonnes conformes.
Année 2022 : Colonnes conformes.
Année 2023 : Colonnes conformes.
Année 2024 : Colonnes conformes.
Année 2025 : Colonnes conformes.


True

In [5]:
#fusion
df_dvf_raw = cd.merge_yearly_dvf(list_dvf)
df_dvf_raw.to_csv("data/merged_raw.csv", index=False)


--- Fusion des bases DVF en cours ---
Fusion terminée. Taille finale : 65634 lignes, 40 colonnes.


Nettoyage de la base


In [6]:
#Nettoyage (projection, gestion des valeurs manquantes et sélection des colonnes utiles et rajout du prix/m²)
gdf_metro = cd.clean_metro_data(gdf_metro_raw)
gdf_dvf = cd.clean_dvf_data(df_dvf_raw)
gdf_dvf['prix_m2'] = gdf_dvf['valeur_fonciere'] / gdf_dvf['surface_reelle_bati']

Analyse et suppression des valeurs extrêmes

In [7]:
seuils_surface=ad.extreme_value_surface(gdf_dvf)

In [8]:
gdf_dvf=cd.remove_extreme_values(gdf_dvf, "surface_reelle_bati", seuils_surface[0], seuils_surface[1])

In [9]:
seuils_prix=ad.extreme_value_prix(gdf_dvf)

In [10]:
gdf_dvf=cd.remove_extreme_values(gdf_dvf, "prix_m2", seuils_prix[0], seuils_prix[1])

In [11]:
#Base fusionnée
gdf_final = cd.merge_dvf_by_line(gdf_dvf, gdf_metro)